# Incremental reviews pipeline

Optional incremental regional update after the primary cleaning pipeline. Select input files explicitly; do not run automatically after a full refresh.

**Execution:** Start Jupyter inside the project tree. Run the path setup first. Review configuration and existing outputs before executing.

In [ ]:
# Resolve the project from the notebook working directory, not a machine-specific path.
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / n).is_dir() for n in ("Python Code", "Data", "Streamlit"))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from inside the AEGEAN project tree.")
SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
CLEANING_DIR = PROJECT_ROOT / "Data" / "Output" / "Cleaning data"
HF_DIR = PROJECT_ROOT / "Data" / "Output" / "Hugging Face"
LLM_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output" / "LLM"
for directory in (CLEANING_DIR, HF_DIR, LLM_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ================================================================
# INCREMENTAL AIRLINE REVIEWS PIPELINE
# ================================================================
# Sources:
#   - TripAdvisor
#   - Trustpilot
#
# Airlines:
#   - Bangkok Airways
#   - Porter Airlines
#   - Virgin Australia
#
# Objective:
#   Process only newly scraped reviews and integrate them into
#   the existing df_final.csv without reprocessing old reviews.
#
# Required packages:
#   pip install pandas numpy tqdm lingua-language-detector openai
# ================================================================

from pathlib import Path
from getpass import getpass
from datetime import datetime
import hashlib
import logging
import os
import re
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from lingua import LanguageDetectorBuilder
from IPython.display import display

tqdm.pandas()

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s"
)

logger = logging.getLogger("airline_incremental_pipeline")


# ================================================================
# 1. CONFIGURATION
# ================================================================

DATA_DIR = SCRAPING_DIR

OUTPUT_DIR = CLEANING_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_FILE = OUTPUT_DIR / "df_final.csv"

CHECKPOINT_FILE = (
    OUTPUT_DIR / "incremental_translation_checkpoint.csv"
)

TRANSLATION_MODEL = "gpt-4.1-mini"

SAVE_EVERY = 25
MAX_TRANSLATION_ATTEMPTS = 4
SLEEP_BETWEEN_CALLS = 0.1

# ------------------------------------------------
# Specify the newly scraped files here.
# Change filenames to match your actual files.
# ------------------------------------------------

NEW_FILES = {
    "TripAdvisor": [
        "tripadvisor_reviews/tripadvisor_bangkok_airways_reviews.csv",
        "tripadvisor_reviews/tripadvisor_porter_reviews.csv",
        "tripadvisor_reviews/tripadvisor_virgin_reviews.csv",
    ],
    "Trustpilot": [
        "trustpilot_reviews/trustpilot_bangkok_airways_reviews.csv",
        "trustpilot_reviews/trustpilot_porter_reviews.csv",
        "trustpilot_reviews/trustpilot_virgin_reviews.csv",
    ],
}

TARGET_AIRLINES = {
    "Bangkok Air",
    "Porter Airlines",
    "Virgin Australia",
}

print("Data directory:", DATA_DIR)
print("Final dataset:", FINAL_FILE)

if not FINAL_FILE.exists():
    raise FileNotFoundError(FINAL_FILE)


# ================================================================
# 2. HELPER FUNCTIONS
# ================================================================

AIRLINE_NAMES = {
    # Aegean
    "aegean": "Aegean Airlines",
    "aegean airlines": "Aegean Airlines",

    # Air France
    "air france": "Air France",

    # Bangkok Airways
    "bangkok": "Bangkok Air",
    "bangkok air": "Bangkok Air",
    "bangkok airways": "Bangkok Air",

    # Lufthansa
    "lufthansa": "Lufthansa",

    # Porter Airlines
    "porter": "Porter Airlines",
    "porter airline": "Porter Airlines",
    "porter airlines": "Porter Airlines",

    # Turkish Airlines
    "turkish": "Turkish Airlines",
    "turkish airlines": "Turkish Airlines",

    # Virgin Australia
    "virgin": "Virgin Australia",
    "virgin australia": "Virgin Australia",
    "virgin australia airlines": "Virgin Australia",
}

AIRLINE_FILENAME_ALIASES = {
    "bangkok": "Bangkok Air",
    "porter": "Porter Airlines",
    "virgin": "Virgin Australia",
}


def clean_text(series):
    """
    Normalize whitespace and missing values.
    Matches the original cleaning pipeline.
    """
    s = (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return s.mask(
        s.str.casefold().isin(
            ["", "nan", "none", "null", "nat"]
        )
    )


def normalize_airline(value):

    """Return or process normalize airline."""
    if pd.isna(value):
        return pd.NA

    name = str(value).strip()

    key = re.sub(
        r"\s+", " ", name.replace("_", " ")
    ).casefold()

    return AIRLINE_NAMES.get(key, name)


def normalize_column(name):

    """Return or process normalize column."""
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(name).casefold()
    )


def find_column(columns, candidates):

    """Return or process find column."""
    lookup = {
        normalize_column(col): col
        for col in columns
    }

    for candidate in candidates:

        actual = lookup.get(
            normalize_column(candidate)
        )

        if actual is not None:
            return actual

    return None


def airline_from_filename(path):

    """Return or process airline from filename."""
    stem = (
        path.stem.casefold()
        .replace("-", "_")
        .replace(" ", "_")
    )

    matches = {
        canonical
        for alias, canonical
        in AIRLINE_FILENAME_ALIASES.items()
        if alias in stem
    }

    if len(matches) == 1:
        return next(iter(matches))

    return None


def parse_dates(series, source):
    """
    Use the original pipeline's mixed-date parsing.
    """
    s = clean_text(series)

    dates = pd.to_datetime(
        s,
        format="mixed",
        errors="coerce",
        utc=True
    )

    missing = dates.isna() & s.notna()

    if missing.any():

        dates.loc[missing] = pd.to_datetime(
            s.loc[missing],
            errors="coerce",
            utc=True,
            dayfirst=True
        )

    return dates.dt.tz_convert(None).dt.normalize()


def parse_ratings(series):

    """Return or process parse ratings."""
    s = clean_text(series).str.replace(
        ",", ".", regex=False
    )

    values = s.str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)(?:\s*/\s*[0-9]+(?:\.[0-9]+)?)?$",
        expand=False
    )

    return pd.to_numeric(
        values,
        errors="coerce"
    )


def read_source_csv(path):
    """
    Read source CSVs with encoding fallback.
    """

    for encoding in ["utf-8-sig", "utf-8", "cp1252"]:

        try:

            return pd.read_csv(
                path,
                encoding=encoding,
                dtype="string",
                low_memory=False
            )

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"Unable to read file: {path}"
    )


# ================================================================
# 3. SOURCE-SPECIFIC COLUMN MAPPINGS
# ================================================================

# Based on the TripAdvisor and Trustpilot samples.

SOURCE_FIELDS = {

    "TripAdvisor": {
        "airline": [
            "locationName",
            "airline",
            "airline_name",
        ],
        "review_date": [
            "review_date",
            "createdDate",
            "publishedAt",
            "publication_date",
            "date",
        ],
        "rating": [
            "overall_rating",
            "rating",
            "bubble_rating",
            "score",
        ],
        "review": [
            "review_text",
            "review",
            "text",
            "content",
            "body",
        ],
    },

    "Trustpilot": {
        "airline": [
            "airline",
            "company",
            "company_name",
        ],
        "review_date": [
            "review_date",
            "publishedAt",
            "createdAt",
            "dateOfExperience",
            "date",
        ],
        "rating": [
            "stars",
            "rating",
            "score",
        ],
        "review": [
            "review",
            "text",
            "review_text",
            "content",
            "body",
        ],
    },
}


COMMON_FIELDS = {
    "airline": [
        "airline",
        "airline_name",
        "locationName",
        "company",
        "company_name",
    ],
    "review_date": [
        "review_date",
        "publication_date",
        "published_at",
        "created_at",
        "date",
    ],
    "rating": [
        "rating",
        "overall_rating",
        "stars",
        "score",
    ],
    "review": [
        "review",
        "review_text",
        "text",
        "content",
        "body",
        "description",
    ],
    "review_id": [
        "source_review_id",
        "review_id",
        "id",
        "reviewId",
        "review_url",
        "url",
    ],
    "title": [
        "review_title",
        "title",
        "headline",
    ],
    "origin": [
        "origin",
        "origin_airport",
        "departure_airport",
        "from",
    ],
    "destination": [
        "destination",
        "destination_airport",
        "arrival_airport",
        "to",
    ],
    "route": [
        "route",
        "flight_route",
    ],
    "flight_number": [
        "flight_number",
        "flight_no",
        "flightNumber",
    ],
}


STANDARD_COLUMNS = [
    "airline",
    "review_date",
    "rating",
    "review",
    "source",
    "review_id",
    "title",
    "origin",
    "destination",
    "route",
    "flight_number",
]


# ================================================================
# 4. STANDARDIZE SOURCE FILE
# ================================================================

def standardize_file(raw, source, path):

    """Return or process standardize file."""
    specific = SOURCE_FIELDS.get(source, {})

    mapped = {}

    for field in STANDARD_COLUMNS:

        candidates = (
            specific.get(field, [])
            + COMMON_FIELDS.get(field, [])
        )

        selected = find_column(
            raw.columns,
            candidates
        )

        if selected is not None:
            mapped[field] = raw[selected]
        else:
            mapped[field] = pd.Series(
                pd.NA,
                index=raw.index,
                dtype="string"
            )

    frame = pd.DataFrame(mapped)

    # Infer airline only if no airline values exist.
    if frame["airline"].isna().all():

        fallback = airline_from_filename(path)

        if fallback:
            frame["airline"] = fallback

    # Validate essential fields.
    if len(raw) > 0:

        if frame["review"].isna().all():
            raise ValueError(
                f"Review text column not recognized: {path.name}"
            )

        if frame["airline"].isna().all():
            raise ValueError(
                f"Airline not identified: {path.name}"
            )

    frame["source"] = source
    frame["source_file"] = path.name

    frame["airline"] = (
        frame["airline"]
        .map(normalize_airline)
        .astype("string")
    )

    for col in [
        "review",
        "review_id",
        "title",
        "origin",
        "destination",
        "route",
        "flight_number",
    ]:

        frame[col] = clean_text(frame[col])

    frame["review_date"] = parse_dates(
        frame["review_date"],
        source
    )

    frame["rating"] = parse_ratings(
        frame["rating"]
    )

    frame["rating_original_scale"] = pd.NA

    return frame


# ================================================================
# 5. STABLE RECORD KEY
# ================================================================

def stable_record_key(row):
    """
    Exactly the same key construction as the original
    pipeline, to allow deduplication against df_final.
    """

    parts = [
        str(row["source"]),
        str(row["airline"])
    ]

    if pd.notna(row["review_id"]):

        parts += [
            "id",
            str(row["review_id"])
        ]

    else:

        date = (
            row["review_date"].strftime("%Y-%m-%d")
            if pd.notna(row["review_date"])
            else ""
        )

        parts += [
            "content",
            date,
            str(row["review"]).casefold()
        ]

    return hashlib.sha256(
        "\x1f".join(parts).encode("utf-8")
    ).hexdigest()


def review_hash(value):

    """Return or process review hash."""
    if pd.isna(value):
        return ""

    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()


# ================================================================
# 6. LOAD EXISTING DATASET
# ================================================================

print("\nLoading existing df_final...")

df_existing = pd.read_csv(
    FINAL_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

df_existing["review_date"] = pd.to_datetime(
    df_existing["review_date"],
    errors="coerce"
)

if "record_key" not in df_existing.columns:
    raise ValueError(
        "Existing df_final has no record_key column."
    )

if df_existing["record_key"].isna().any():
    raise ValueError("Missing existing record keys")

if df_existing["record_key"].duplicated().any():
    raise ValueError("Duplicate existing record keys")

print(
    "Existing records:",
    f"{len(df_existing):,}"
)


# ================================================================
# 7. LOAD NEW TRIPADVISOR AND TRUSTPILOT FILES
# ================================================================

new_frames = []

ingestion_log = []

for source, filenames in NEW_FILES.items():

    for filename in filenames:

        path = DATA_DIR / filename
        if not path.exists():
            aliases = {"bangkok_airways": "bangkok_airways", "porter": "porter_airlines", "virgin": "virgin_australia"}
            stem = Path(filename).stem
            prefix, _, airline = stem.partition("_")
            airline = airline.removesuffix("_reviews")
            path = path.parent / (aliases.get(airline, airline) + "_reviews.csv")

        if not path.exists():
            raise FileNotFoundError(path)

        raw = read_source_csv(path)

        standardized = standardize_file(
            raw,
            source,
            path
        )

        unexpected = (
            set(
                standardized["airline"]
                .dropna()
                .unique()
            )
            - TARGET_AIRLINES
        )

        if unexpected:
            raise ValueError(
                f"Unexpected airlines in {filename}: "
                f"{unexpected}"
            )

        new_frames.append(standardized)

        ingestion_log.append({
            "source": source,
            "file": filename,
            "records": len(standardized),
            "missing_reviews": int(
                standardized["review"].isna().sum()
            ),
            "missing_dates": int(
                standardized["review_date"].isna().sum()
            )
        })

        print(
            f"Loaded {source}: {filename} "
            f"({len(standardized):,} records)"
        )

if not new_frames:
    raise ValueError("No new files loaded")

df_new = pd.concat(
    new_frames,
    ignore_index=True
)

print(
    "\nTotal newly scraped records:",
    f"{len(df_new):,}"
)

display(pd.DataFrame(ingestion_log))


# ================================================================
# 8. CLEAN AND REMOVE DUPLICATES
# ================================================================

df_new["review"] = clean_text(
    df_new["review"]
)

df_new["airline"] = clean_text(
    df_new["airline"]
)

df_new["record_key"] = df_new.apply(
    stable_record_key,
    axis=1
)

before = len(df_new)

df_new = df_new.drop_duplicates(
    subset=["record_key"],
    keep="first"
).copy()

print(
    "\nDuplicates within new files:",
    before - len(df_new)
)

existing_keys = set(
    df_existing["record_key"].astype(str)
)

df_new = df_new.loc[
    ~df_new["record_key"].astype(str).isin(
        existing_keys
    )
].copy()

df_new.reset_index(
    drop=True,
    inplace=True
)

df_new["has_review"] = (
    df_new["review"].notna()
)

print(
    "Genuinely new records:",
    f"{len(df_new):,}"
)

print(
    "New records with text:",
    f"{df_new['has_review'].sum():,}"
)

print(
    "New records without text:",
    f"{(~df_new['has_review']).sum():,}"
)


# ================================================================
# 9. LANGUAGE DETECTION
# ================================================================

detector = (
    LanguageDetectorBuilder
    .from_all_languages()
    .build()
)


def detect_language_with_confidence(value):

    """Return or process detect language with confidence."""
    if pd.isna(value) or not str(value).strip():

        return pd.Series({
            "original_language": None,
            "language_confidence": None
        })

    values = detector.compute_language_confidence_values(
        str(value)
    )

    if not values:

        return pd.Series({
            "original_language": None,
            "language_confidence": None
        })

    best = values[0]

    return pd.Series({
        "original_language": best.language.name.title(),
        "language_confidence": float(best.value)
    })


if not df_new.empty:

    print("\nDetecting original languages...")

    language_results = (
        df_new["review"]
        .progress_apply(detect_language_with_confidence)
    )

    df_new = pd.concat(
        [
            df_new,
            language_results
        ],
        axis=1
    )

    df_new["language"] = (
        df_new["original_language"]
    )

    print("\nLanguage distribution:")

    display(
        df_new["original_language"]
        .value_counts(dropna=False)
    )


# ================================================================
# 10. INITIALIZE TRANSLATION COLUMNS
# ================================================================

if not df_new.empty:

    df_new["review_hash"] = (
        df_new["review"].map(review_hash)
    )

    df_new["review_en"] = (
        df_new["review"].copy()
    )

    df_new["translated"] = False

    df_new["translation_status"] = "pending"

    empty_mask = ~df_new["has_review"]

    english_mask = (
        df_new["original_language"].eq("English")
        & df_new["has_review"]
    ).fillna(False)

    df_new.loc[
        empty_mask,
        "review_en"
    ] = pd.NA

    df_new.loc[
        empty_mask,
        "translation_status"
    ] = "empty"

    df_new.loc[
        english_mask,
        "translation_status"
    ] = "original_english"


# ================================================================
# 11. RESTORE TRANSLATION CHECKPOINT
# ================================================================

if (
    not df_new.empty
    and CHECKPOINT_FILE.exists()
):

    print("\nRestoring translation checkpoint...")

    previous = pd.read_csv(
        CHECKPOINT_FILE,
        dtype="string"
    )

    required = {
        "record_key",
        "review_hash",
        "review_en",
        "translation_status"
    }

    if required.issubset(previous.columns):

        previous = (
            previous
            .drop_duplicates("record_key", keep="last")
            .set_index("record_key")
        )

        restored = 0

        for idx in df_new.index:

            key = df_new.at[
                idx,
                "record_key"
            ]

            if key not in previous.index:
                continue

            old = previous.loc[key]

            same_text = (
                old["review_hash"]
                == df_new.at[idx, "review_hash"]
            )

            successful = (
                old["translation_status"]
                in ["completed", "original_english"]
            )

            if (
                same_text
                and successful
                and pd.notna(old["review_en"])
            ):

                df_new.at[
                    idx, "review_en"
                ] = old["review_en"]

                df_new.at[
                    idx, "translation_status"
                ] = old["translation_status"]

                df_new.at[
                    idx, "translated"
                ] = (
                    old["translation_status"]
                    == "completed"
                )

                restored += 1

        print(
            "Restored translations:",
            f"{restored:,}"
        )


def save_checkpoint():

    """Return or process save checkpoint."""
    if df_new.empty:
        return

    columns = [
        "record_key",
        "review_hash",
        "review_en",
        "translation_status",
        "translated"
    ]

    temporary = CHECKPOINT_FILE.with_suffix(
        ".tmp"
    )

    df_new[columns].to_csv(
        temporary,
        index=False,
        encoding="utf-8-sig"
    )

    temporary.replace(CHECKPOINT_FILE)


# ================================================================
# 12. TRANSLATE NON-ENGLISH REVIEWS
# ================================================================

pending_indices = (
    df_new.index[
        df_new["translation_status"].isin(
            ["pending", "error"]
        )
    ]
    if not df_new.empty
    else []
)

print(
    "\nReviews requiring translation:",
    f"{len(pending_indices):,}"
)

if len(pending_indices) > 0:

    from openai import OpenAI

    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        api_key = getpass(
            "Enter OpenAI API key: "
        )

    client = OpenAI(
        api_key=api_key,
        timeout=90.0,
        max_retries=2
    )

    # Reuse translations for identical review texts.
    translation_cache = {}

    # Load successful translations from the current batch.
    for row in df_new.itertuples():

        if (
            row.translation_status == "completed"
            and pd.notna(row.review_en)
        ):

            translation_cache[
                str(row.review)
            ] = str(row.review_en)

    # Also reuse identical texts already translated
    # in the existing dataset.
    if {
        "review",
        "review_en",
        "translation_status"
    }.issubset(df_existing.columns):

        old_translated = df_existing.loc[
            df_existing["translation_status"].eq(
                "completed"
            )
            & df_existing["review"].notna()
            & df_existing["review_en"].notna(),
            ["review", "review_en"]
        ]

        for row in old_translated.itertuples(
            index=False
        ):

            translation_cache[
                str(row.review)
            ] = str(row.review_en)

    def translate_review(text, language):

        """Return or process translate review."""
        response = client.responses.create(
            model=TRANSLATION_MODEL,
            instructions=(
                "Translate the following airline passenger "
                "review into natural English. "
                f"Detected language: {language}. "
                "Preserve the original meaning, sentiment, "
                "criticism, sarcasm, names, numbers, "
                "airport codes and flight details. "
                "Do not summarize, censor, or add information. "
                "Return only the translated review."
            ),
            input=str(text),
            max_output_tokens=4000
        )

        result = (
            response.output_text or ""
        ).strip()

        if not result:
            raise ValueError(
                "Empty translation response"
            )

        return result

    try:

        for counter, idx in enumerate(
            tqdm(
                pending_indices,
                desc="Translating reviews"
            ),
            start=1
        ):

            original = str(
                df_new.at[idx, "review"]
            )

            language = df_new.at[
                idx, "original_language"
            ]

            try:

                if original in translation_cache:

                    translated_text = (
                        translation_cache[original]
                    )

                else:

                    for attempt in range(
                        MAX_TRANSLATION_ATTEMPTS
                    ):

                        try:

                            translated_text = (
                                translate_review(
                                    original,
                                    language
                                )
                            )

                            break

                        except Exception:

                            if (
                                attempt
                                == MAX_TRANSLATION_ATTEMPTS - 1
                            ):
                                raise

                            time.sleep(
                                min(
                                    2 ** (attempt + 1),
                                    30
                                )
                            )

                    translation_cache[
                        original
                    ] = translated_text

                    time.sleep(
                        SLEEP_BETWEEN_CALLS
                    )

                df_new.at[
                    idx, "review_en"
                ] = translated_text

                df_new.at[
                    idx, "translated"
                ] = True

                df_new.at[
                    idx, "translation_status"
                ] = "completed"

            except Exception as exc:

                df_new.at[
                    idx, "translation_status"
                ] = "error"

                logger.error(
                    "Translation failed: %s | %s",
                    df_new.at[idx, "record_key"],
                    exc
                )

            if counter % SAVE_EVERY == 0:
                save_checkpoint()

    finally:
        # Save even if interrupted manually.
        save_checkpoint()


# ================================================================
# 13. VALIDATE NEW REVIEWS
# ================================================================

print("\n" + "=" * 60)
print("INCREMENTAL DATA QUALITY REPORT")
print("=" * 60)

if not df_new.empty:

    missing_dates = (
        df_new["review_date"].isna().sum()
    )

    missing_airlines = (
        df_new["airline"].isna().sum()
    )

    failed_translations = (
        df_new["translation_status"]
        .isin(["pending", "error"])
        .sum()
    )

    missing_english = (
        df_new["has_review"]
        & (
            df_new["review_en"].isna()
            | df_new["review_en"].astype("string")
              .str.strip().eq("").fillna(True)
        )
    ).sum()

    print("New records:", len(df_new))
    print("Missing dates:", missing_dates)
    print("Missing airlines:", missing_airlines)
    print("Failed translations:", failed_translations)
    print("Missing English review text:", missing_english)

    print("\nTranslation status:")

    display(
        df_new["translation_status"]
        .value_counts(dropna=False)
    )

    if (
        missing_dates > 0
        or missing_airlines > 0
        or failed_translations > 0
        or missing_english > 0
    ):

        # Save unmerged data for inspection.
        audit_file = (
            OUTPUT_DIR / "new_reviews_pending_validation.csv"
        )

        df_new.to_csv(
            audit_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise RuntimeError(
            "Validation failed. "
            f"Inspect {audit_file}. "
            "Existing df_final.csv was not modified."
        )


# ================================================================
# 14. MERGE NEW REVIEWS INTO df_final
# ================================================================

print("\nMerging new reviews...")

all_columns = list(
    dict.fromkeys(
        list(df_existing.columns)
        + list(df_new.columns)
    )
)

df_final = pd.concat(
    [
        df_existing.reindex(
            columns=all_columns
        ),
        df_new.reindex(
            columns=all_columns
        )
    ],
    ignore_index=True
)

if df_final["record_key"].duplicated().any():

    raise ValueError(
        "Duplicate record keys after merge"
    )

assert (
    len(df_final)
    == len(df_existing) + len(df_new)
)

print(
    "Existing records:",
    f"{len(df_existing):,}"
)

print(
    "New records:",
    f"{len(df_new):,}"
)

print(
    "Combined records:",
    f"{len(df_final):,}"
)


# ================================================================
# 15. SAVE UPDATED DATASET
# ================================================================

# Create a timestamped backup before replacing the file.
if len(df_new) > 0:

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    backup_file = (
        OUTPUT_DIR
        / f"df_final_backup_{timestamp}.csv"
    )

    import shutil

    shutil.copy2(
        FINAL_FILE,
        backup_file
    )

    print(
        "\nBackup created:",
        backup_file
    )

    temporary_file = (
        OUTPUT_DIR / "df_final_incremental.tmp"
    )

    df_final.to_csv(
        temporary_file,
        index=False,
        encoding="utf-8-sig"
    )

    temporary_file.replace(FINAL_FILE)

    print(
        "Updated dataset saved:",
        FINAL_FILE
    )

else:

    print(
        "\nNo new records. Existing dataset unchanged."
    )


# ================================================================
# 16. FINAL DATA QUALITY REPORT
# ================================================================

print("\n" + "=" * 60)
print("FINAL DATASET REPORT")
print("=" * 60)

print(
    "Total rows:",
    f"{len(df_final):,}"
)

print(
    "Non-empty review text:",
    f"{df_final['review'].notna().sum():,}"
)

print(
    "Empty reviews:",
    f"{df_final['review'].isna().sum():,}"
)

print(
    "Untranslated non-English reviews:",
    df_final["translation_status"]
    .isin(["pending", "error"])
    .sum()
)

print(
    "Unknown dates:",
    df_final["review_date"].isna().sum()
)

print(
    "Unknown airlines:",
    df_final["airline"].isna().sum()
)


# ================================================================
# 17. NEW REVIEWS BY AIRLINE AND SOURCE
# ================================================================

print("\nNEW REVIEWS BY AIRLINE AND SOURCE")

if not df_new.empty:

    display(
        pd.crosstab(
            df_new["airline"],
            df_new["source"],
            margins=True,
            margins_name="Total"
        )
    )


# ================================================================
# 18. UPDATED DATASET SUMMARY
# ================================================================

def review_summary(data):

    """Return or process review summary."""
    subset = data.loc[
        data["review"].notna()
    ]

    if subset.empty:
        return pd.DataFrame()

    return pd.crosstab(
        subset["airline"],
        subset["source"],
        margins=True,
        margins_name="Total"
    )


print("\nALL REVIEWS")

table_all = review_summary(df_final)

display(table_all)


print("\nREVIEWS FROM 2024 ONWARDS")

table_2024 = review_summary(
    df_final.loc[
        df_final["review_date"]
        >= pd.Timestamp("2024-01-01")
    ]
)

display(table_2024)


print("\nREVIEWS FROM 2026 ONWARDS")

table_2026 = review_summary(
    df_final.loc[
        df_final["review_date"]
        >= pd.Timestamp("2026-01-01")
    ]
)

display(table_2026)


# ================================================================
# 19. EXPORT SUMMARY TABLES
# ================================================================

summary_file = (
    OUTPUT_DIR / "airline_review_summary_updated.xlsx"
)

with pd.ExcelWriter(
    summary_file,
    engine="openpyxl"
) as writer:

    table_all.to_excel(
        writer,
        sheet_name="All Reviews"
    )

    table_2024.to_excel(
        writer,
        sheet_name="From 2024"
    )

    table_2026.to_excel(
        writer,
        sheet_name="From 2026"
    )

    pd.DataFrame(
        ingestion_log
    ).to_excel(
        writer,
        sheet_name="New Files Audit",
        index=False
    )

print("\nSummary saved:", summary_file)

print("\nPIPELINE COMPLETED SUCCESSFULLY")

In [ ]:
df_final.tail()

In [ ]:
df_final.info()
